<a href="https://colab.research.google.com/github/fathinahnj/unet-plankton-segmentation/blob/main/kfold_as_default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ini coba2 saja 5-fold cross validation in an entirely new file. kita coba pake model baseline dulu. kalo ini berhasil, kita coba juga model yg augmented. kalo misal oke, jadikan ini sebagai default instead of the old files.

kalo nda berhasil, kita kembali ke old files, kita selipkan disitu kodenya

# Imports

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Vjof9rBzJ2QeB82ApBRi")
project = rf.workspace("planktonclassification").project("instance-t20")
version = project.version(4)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...


In [1]:
%%capture
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd

from pycocotools.coco import COCO
from sklearn.model_selection import KFold

from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, Model

from tensorflow.keras.applications import (
    ResNet50,
    EfficientNetB4,
    MobileNetV2
)

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


# Path & Config

In [4]:
BASE_PATH = "/content/drive/MyDrive/PLANKTON/data/instance/test"
data_path = "/content/instance-t20-4/valid"

In [5]:
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 20
KFOLD = 5
LR = 1e-5

MODEL_SAVE_DIR = os.path.join(BASE_PATH, "kfold_models")

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [6]:
ENCODERS = {

    "resnet50": ResNet50,
    "efficientnetb4": EfficientNetB4,
    "mobilenetv2": MobileNetV2
}

# Load COCO Dataset

In [7]:
coco_json_path = os.path.join(data_path, "_annotations.coco.json")

coco = COCO(coco_json_path)

all_images = coco.dataset["images"]
all_annotations = coco.dataset["annotations"]
categories = coco.dataset["categories"]

image_ids = np.array([img["id"] for img in all_images])

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [8]:
coco_dataset = {
    "images": all_images,
    "annotations": all_annotations,
    "categories": categories
}

coco_kfold_obj = COCO()
coco_kfold_obj.dataset = coco_dataset
coco_kfold_obj.createIndex()

creating index...
index created!


# Image Path

In [9]:
def get_image_path(file_name):
    return os.path.join(data_path, file_name)

# Mask Creation

In [10]:
def create_mask(image_info, coco_instance):

    height = image_info["height"]
    width = image_info["width"]

    mask = np.zeros((height, width), dtype=np.uint8)

    ann_ids = coco_instance.getAnnIds(imgIds=image_info["id"])
    anns = coco_instance.loadAnns(ann_ids)

    for ann in anns:
        m = coco_instance.annToMask(ann)
        mask[m == 1] = 1

    return mask

In [11]:
def load_image_and_mask(img_info):

    img_path = os.path.join(data_path, img_info["file_name"])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0

    mask = create_mask(img_info, coco_kfold_obj)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE),
                      interpolation=cv2.INTER_NEAREST)

    mask = np.expand_dims(mask, axis=-1)

    return image, mask

# Dice Score

In [12]:
def dice_score(y_true, y_pred):

    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred)

    return (2. * intersection + 1e-7) / (union + 1e-7)

# Loss Function

In [13]:
def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2. * intersection + 1e-7) / (union + 1e-7)

    return bce + (1 - dice)

# UNet Builder + Encoders

In [14]:
def build_unet_with_encoder(encoder_name):

    EncoderClass = ENCODERS[encoder_name]

    encoder = EncoderClass(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    skips = []

    if encoder_name == "resnet50":
        skips = [
            encoder.get_layer("conv1_relu").output,
            encoder.get_layer("conv2_block3_out").output,
            encoder.get_layer("conv3_block4_out").output,
            encoder.get_layer("conv4_block6_out").output
        ]
        bridge = encoder.get_layer("conv5_block3_out").output

    elif encoder_name == "efficientnetb4":
        skips = [
            encoder.get_layer("block2a_activation").output,
            encoder.get_layer("block3a_activation").output,
            encoder.get_layer("block4a_activation").output,
            encoder.get_layer("block6a_activation").output
        ]
        bridge = encoder.get_layer("top_activation").output

    elif encoder_name == "mobilenetv2":
        skips = [
            encoder.get_layer("block_1_expand_relu").output,
            encoder.get_layer("block_3_expand_relu").output,
            encoder.get_layer("block_6_expand_relu").output,
            encoder.get_layer("block_13_expand_relu").output
        ]
        bridge = encoder.get_layer("block_16_project").output

    x = bridge

    for skip in reversed(skips):

        x = layers.UpSampling2D(size=(2, 2))(x)
        x = layers.Concatenate()([x, skip])
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)

    # Add an additional upsampling block to match IMG_SIZE (224x224)
    x = layers.UpSampling2D(size=(2, 2))(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(x)

    model = Model(encoder.input, outputs)

    return model

# K-Fold Setup

In [15]:
kf = KFold(
    n_splits=KFOLD,
    shuffle=True,
    random_state=42
)

encoder_results = {}

# Train & Val

In [16]:
for encoder_name in ENCODERS.keys():

    print(f"\n==============================")
    print(f"TRAINING ENCODER: {encoder_name}")
    print(f"==============================")

    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(image_ids)):

        print(f"\n----- Fold {fold+1} -----")

        train_ids = image_ids[train_idx]
        val_ids = image_ids[val_idx]

        model = build_unet_with_encoder(encoder_name)

        model.compile(
            optimizer=Adam(LR),
            loss=bce_dice_loss
        )

        train_images = []
        train_masks = []

        for img in all_images:

            if img["id"] in train_ids:
                image, mask = load_image_and_mask(img)
                train_images.append(image)
                train_masks.append(mask)

        train_images = np.array(train_images)
        train_masks = np.array(train_masks)

        model.fit(
            train_images,
            train_masks,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=1
        )

        fold_dice = []

        for img in all_images:

            if img["id"] in val_ids:

                image, mask_true = load_image_and_mask(img)

                image = np.expand_dims(image, axis=0)

                pred = model.predict(image, verbose=0)
                pred = (pred > 0.3).astype(np.uint8)[0,:,:,0]

                d = dice_score(mask_true[:,:,0], pred)
                fold_dice.append(d)

        mean_dice = np.mean(fold_dice)
        fold_scores.append(mean_dice)

        print("Fold Dice:", mean_dice)

        model.save(os.path.join(
            MODEL_SAVE_DIR,
            f"{encoder_name}_fold_{fold+1}.h5"
        ))

    encoder_results[encoder_name] = fold_scores


TRAINING ENCODER: resnet50

----- Fold 1 -----
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 184s 5s/step - loss: 1.2454
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.9035
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.4805
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 522ms/step - loss: 0.4133
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.3031
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2834
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2548
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2530
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 494ms/step - loss: 0.2295
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - loss: 0.2294
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2030
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.1858
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.1677
Epoch 14/20
17/17 ━━━━━━━━━

Fold Dice: 6.015463701462838e-11

----- Fold 2 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - loss: 1.1383
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 495ms/step - loss: 0.7032
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.3992
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 525ms/step - loss: 0.3588
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.3008
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.2613
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 500ms/step - loss: 0.2682
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2307
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.2403
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 491ms/step - loss: 0.2352
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.2223
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2010
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 501ms/step - loss: 0.1996
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.1806
Epoc

Fold Dice: 4.71517452621683e-11

----- Fold 3 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - loss: 1.3175
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 500ms/step - loss: 1.0965
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 515ms/step - loss: 0.6719
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 525ms/step - loss: 0.4525
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.3325
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2902
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 497ms/step - loss: 0.2611
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 488ms/step - loss: 0.2479
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 492ms/step - loss: 0.2289
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.2217
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - loss: 0.2319
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2219
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.1953
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 509ms/step - loss: 0.1929
Epoc

Fold Dice: 4.802275441775151e-11

----- Fold 4 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 140s 5s/step - loss: 1.2596
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 482ms/step - loss: 0.9795
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 493ms/step - loss: 0.5302
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.4031
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.4171
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 529ms/step - loss: 0.3311
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 518ms/step - loss: 0.2925
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.2556
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 502ms/step - loss: 0.2742
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 496ms/step - loss: 0.2324
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 498ms/step - loss: 0.2248
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 499ms/step - loss: 0.1964
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2032
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 506ms/step - loss: 0.1860
Epoch

Fold Dice: 5.904451968690509e-11

----- Fold 5 -----
Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - loss: 1.2703
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.9397
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 517ms/step - loss: 0.4942
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 528ms/step - loss: 0.4021
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 524ms/step - loss: 0.4143
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.3299
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 504ms/step - loss: 0.2969
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 499ms/step - loss: 0.2700
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 497ms/step - loss: 0.2390
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 501ms/step - loss: 0.2463
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 505ms/step - loss: 0.2660
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 507ms/step - loss: 0.2321
Epoch 13/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 515ms/step - loss: 0.2000
Epoch 14/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 508ms/step - loss: 0.1861
Epoch

Fold Dice: 5.7022476103822407e-11

TRAINING ENCODER: efficientnetb4

----- Fold 1 -----
71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 14, 14, 1792), (None, 7, 7, 960)]

# Result Comparison

In [ ]:
comparison = []

for encoder, scores in encoder_results.items():

    comparison.append({
        "Encoder": encoder,
        "Fold1": scores[0],
        "Fold2": scores[1],
        "Fold3": scores[2],
        "Fold4": scores[3],
        "Fold5": scores[4],
        "Average Dice": np.mean(scores),
        "Std Dice": np.std(scores)
    })

df = pd.DataFrame(comparison)
df = df.sort_values("Average Dice", ascending=False)

df